In [ ]:
"""
EDUCATIONAL FOOTBALL PREDICTOR v5.0 - ENHANCED EDITION
=======================================================
Now with advanced features and market comparison!

Uses comprehensive data from football-data.co.uk including:
- Shot accuracy (HST/HS)
- Corners, cards, fouls
- Real bookmaker odds for comparison
- Market efficiency analysis

EDUCATIONAL PURPOSE ONLY - Learn why markets are hard to beat

Author: Educational Tool
Date: October 2025
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss
import warnings
import os
import json
from datetime import datetime
import re

try:
    import tkinter as tk
    from tkinter import ttk, scrolledtext, messagebox
    GUI_AVAILABLE = True
except ImportError:
    GUI_AVAILABLE = False

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

CONFIG = {
    'leagues': {
        'E0': 'Premier League',
        'SP1': 'La Liga',
        'D1': 'Bundesliga',
        'I1': 'Serie A',
        'F1': 'Ligue 1',
    },
    'seasons': ['2425', '2324', '2223', '2122', '2021'],
    'base_url': 'https://www.football-data.co.uk/mmz4281',
    'data_folder': 'football_data',
    'log_folder': 'prediction_logs',
    'form_window': 5,
    'random_state': 42,
    'min_logs_required': 5,
}

TEAM_ALIASES = {
    'Man City': 'Manchester City', 'Man Utd': 'Manchester United',
    'Man United': 'Manchester United', 'Spurs': 'Tottenham',
    'Newcastle': 'Newcastle United', 'West Ham': 'West Ham United',
    'Wolves': 'Wolverhampton Wanderers', 'Brighton': 'Brighton & Hove Albion',
}

DISCLAIMER = """
╔══════════════════════════════════════════════════════════════╗
║     EDUCATIONAL TOOL v5.0 - ENHANCED WITH MARKET DATA        ║
╚══════════════════════════════════════════════════════════════╝

NEW IN v5.0:
• Advanced features: shot accuracy, corners, discipline
• Real bookmaker odds comparison (Bet365, Pinnacle)
• Market efficiency analysis
• See WHY your model can't beat the market

LEARNING GOALS:
✓ Compare model accuracy vs bookmaker accuracy
✓ Understand efficient market hypothesis
✓ See bookmaker margins (5-7%) eliminate edges
✓ Learn feature importance (what really matters?)

This demonstrates why sports betting is mathematically 
challenging - even good models struggle against markets.

Problem gambling help: 1-800-522-4700
══════════════════════════════════════════════════════════════
"""

# ============================================================================
# UTILITIES
# ============================================================================

def ensure_folders():
    for folder in [CONFIG['data_folder'], CONFIG['log_folder']]:
        if not os.path.exists(folder):
            os.makedirs(folder)

def normalize_team(team):
    return TEAM_ALIASES.get(team.strip(), team.strip())

# ============================================================================
# LOGGING SYSTEM (Enhanced with market comparison)
# ============================================================================

class PredictionLogger:
    def __init__(self):
        self.log_file = f"{CONFIG['log_folder']}/predictions_v5.json"
        self.logs = self.load_logs()
    
    def load_logs(self):
        if os.path.exists(self.log_file):
            try:
                with open(self.log_file, 'r') as f:
                    return json.load(f)
            except:
                return []
        return []
    
    def save_logs(self):
        with open(self.log_file, 'w') as f:
            json.dump(self.logs, f, indent=2)
    
    def add_log(self, prediction, actual):
        """Log with market comparison"""
        log = {
            'timestamp': datetime.now().isoformat(),
            'match': f"{prediction['home']} vs {prediction['away']}",
            'model_predicted': self._get_top(prediction['probabilities']),
            'model_probs': prediction['probabilities'],
            'market_probs': prediction.get('market_probs', {}),
            'actual': actual,
            'model_correct': self._check(prediction, actual),
            'market_correct': self._check_market(prediction, actual)
        }
        self.logs.append(log)
        self.save_logs()
    
    def _get_top(self, probs):
        return max(probs.items(), key=lambda x: x[1])[0]
    
    def _check(self, prediction, actual):
        top = self._get_top(prediction['probabilities'])
        result_map = {'H': 'home', 'D': 'draw', 'A': 'away'}
        return top == result_map.get(actual.get('result', ''), 'unknown')
    
    def _check_market(self, prediction, actual):
        if not prediction.get('market_probs'):
            return None
        top = self._get_top(prediction['market_probs'])
        result_map = {'H': 'home', 'D': 'draw', 'A': 'away'}
        return top == result_map.get(actual.get('result', ''), 'unknown')
    
    def get_stats(self):
        if not self.logs:
            return {
                'total': 0, 'model_correct': 0, 'market_correct': 0,
                'model_accuracy': 0, 'market_accuracy': 0,
                'needs_more': CONFIG['min_logs_required']
            }
        
        model_correct = sum(1 for log in self.logs if log.get('model_correct'))
        market_correct = sum(1 for log in self.logs if log.get('market_correct') == True)
        market_total = sum(1 for log in self.logs if log.get('market_correct') is not None)
        total = len(self.logs)
        
        return {
            'total': total,
            'model_correct': model_correct,
            'model_accuracy': model_correct / total if total > 0 else 0,
            'market_correct': market_correct,
            'market_accuracy': market_correct / market_total if market_total > 0 else 0,
            'market_available': market_total,
            'needs_more': max(0, CONFIG['min_logs_required'] - total)
        }
    
    def can_generate(self):
        return len(self.logs) >= CONFIG['min_logs_required']

# ============================================================================
# PARSING
# ============================================================================

def parse_fixtures(text):
    fixtures = []
    lines = text.strip().split('\n')
    
    for line in lines:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        
        if '|' in line and ' vs ' in line:
            parts = [p.strip() for p in line.split('|')]
            if len(parts) >= 1:
                teams = parts[0].split(' vs ')
                if len(teams) == 2:
                    fixtures.append({
                        'HomeTeam': normalize_team(teams[0]),
                        'AwayTeam': normalize_team(teams[1]),
                        'Competition': parts[1] if len(parts) > 1 else 'Unknown',
                        'Kickoff': parts[2] if len(parts) > 2 else ''
                    })
        elif ' vs ' in line:
            teams = line.split(' vs ')
            if len(teams) == 2:
                fixtures.append({
                    'HomeTeam': normalize_team(teams[0]),
                    'AwayTeam': normalize_team(teams[1]),
                    'Competition': 'Unknown',
                    'Kickoff': ''
                })
    
    return fixtures

def parse_odds(text, fixtures):
    odds_map = {}
    lines = text.strip().split('\n')
    
    for line in lines:
        if ':' not in line:
            continue
        
        try:
            match_name, odds_str = line.split(':', 1)
            odds_parts = [float(x.strip()) for x in odds_str.split('|')]
            
            if len(odds_parts) >= 5:
                odds_map[match_name.strip()] = {
                    'home_win': odds_parts[0],
                    'draw': odds_parts[1],
                    'away_win': odds_parts[2],
                    'over_2.5': odds_parts[3],
                    'btts_yes': odds_parts[4]
                }
        except:
            continue
    
    for fixture in fixtures:
        match_key = f"{fixture['HomeTeam']} vs {fixture['AwayTeam']}"
        fixture['odds'] = odds_map.get(match_key, None)
    
    return fixtures

def calc_implied_probs(odds):
    """Calculate implied probabilities from odds"""
    if not odds:
        return None
    
    implied = {
        'home': 1 / odds['home_win'],
        'draw': 1 / odds['draw'],
        'away': 1 / odds['away_win']
    }
    
    overround = implied['home'] + implied['draw'] + implied['away']
    
    # Normalize to remove bookmaker margin
    total = implied['home'] + implied['draw'] + implied['away']
    implied['home_norm'] = implied['home'] / total
    implied['draw_norm'] = implied['draw'] / total
    implied['away_norm'] = implied['away'] / total
    implied['overround_pct'] = (overround - 1.0) * 100
    
    return implied

# ============================================================================
# DATA LOADING
# ============================================================================

def download_data(leagues, seasons):
    print("=" * 80)
    print("LOADING HISTORICAL DATA (Enhanced with stats & odds)")
    print("=" * 80)
    
    all_data = []
    
    for league in leagues:
        print(f"\n{CONFIG['leagues'].get(league, league)}")
        
        for season in seasons:
            filename = f"{CONFIG['data_folder']}/{league}_{season}.csv"
            
            if os.path.exists(filename):
                print(f"  {season}: Cached")
                df = pd.read_csv(filename, encoding='latin-1', encoding_errors='ignore')
            else:
                url = f"{CONFIG['base_url']}/{season}/{league}.csv"
                try:
                    print(f"  {season}: Downloading...")
                    df = pd.read_csv(url, encoding='latin-1', encoding_errors='ignore')
                    df.to_csv(filename, index=False)
                except Exception as e:
                    print(f"  Failed: {e}")
                    continue
            
            df['Season'] = season
            df['League'] = league
            all_data.append(df)
    
    if not all_data:
        raise Exception("No data loaded")
    
    combined = pd.concat(all_data, ignore_index=True)
    print(f"\nTotal: {len(combined)} matches")
    
    # Show available columns
    stats_cols = [c for c in combined.columns if c in ['HST', 'AST', 'HC', 'AC', 'HY', 'AY']]
    odds_cols = [c for c in combined.columns if c in ['B365H', 'PSH', 'AvgH', 'Avg>2.5']]
    print(f"Stats available: {', '.join(stats_cols) if stats_cols else 'None'}")
    print(f"Odds available: {', '.join(odds_cols) if odds_cols else 'None'}")
    
    return combined

def update_current_season(leagues):
    print("\nUpdating current season...")
    for league in leagues:
        filename = f"{CONFIG['data_folder']}/{league}_2425.csv"
        url = f"{CONFIG['base_url']}/2425/{league}.csv"
        try:
            df = pd.read_csv(url, encoding='latin-1', encoding_errors='ignore')
            df.to_csv(filename, index=False)
            print(f"  {CONFIG['leagues'][league]}: {len(df)} matches")
        except Exception as e:
            print(f"  {CONFIG['leagues'][league]}: Failed")

def preprocess_data(df):
    """Enhanced preprocessing with stats and odds"""
    
    # Essential columns
    essential = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'Season', 'League']
    
    # Optional stats columns
    stats = ['HST', 'AST', 'HS', 'AS', 'HC', 'AC', 'HY', 'AY', 'HF', 'AF']
    
    # Optional odds columns (Bet365 preferred, then Pinnacle, then average)
    odds = ['B365H', 'B365D', 'B365A', 'PSH', 'PSD', 'PSA', 
            'AvgH', 'AvgD', 'AvgA', 'Avg>2.5', 'Avg<2.5']
    
    # Use what's available
    cols_to_use = essential + [c for c in stats if c in df.columns] + [c for c in odds if c in df.columns]
    df = df[cols_to_use].copy()
    
    df = df.dropna(subset=['FTHG', 'FTAG', 'FTR'])
    
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='coerce')
        df = df.dropna(subset=['Date'])
    
    df = df.sort_values('Date').reset_index(drop=True)
    
    # Targets
    df['TotalGoals'] = df['FTHG'] + df['FTAG']
    df['Over2.5'] = (df['TotalGoals'] > 2.5).astype(int)
    df['BTTS'] = ((df['FTHG'] > 0) & (df['FTAG'] > 0)).astype(int)
    
    return df

# ============================================================================
# ENHANCED FEATURE ENGINEERING
# ============================================================================

def calc_form(df, team, n=5):
    matches = df[(df['HomeTeam'] == team) | (df['AwayTeam'] == team)].copy()
    if len(matches) < n:
        return 5
    recent = matches.tail(n)
    points = 0
    for _, m in recent.iterrows():
        if m['HomeTeam'] == team:
            points += 3 if m['FTR'] == 'H' else (1 if m['FTR'] == 'D' else 0)
        else:
            points += 3 if m['FTR'] == 'A' else (1 if m['FTR'] == 'D' else 0)
    return points

def calc_goals(df, team, n=5, loc='both'):
    if loc == 'home':
        m = df[df['HomeTeam'] == team].tail(n)
        scored = m['FTHG'].mean() if len(m) > 0 else 1.5
        conceded = m['FTAG'].mean() if len(m) > 0 else 1.0
    elif loc == 'away':
        m = df[df['AwayTeam'] == team].tail(n)
        scored = m['FTAG'].mean() if len(m) > 0 else 1.2
        conceded = m['FTHG'].mean() if len(m) > 0 else 1.3
    else:
        home = df[df['HomeTeam'] == team].tail(n)
        away = df[df['AwayTeam'] == team].tail(n)
        total = len(home) + len(away)
        scored = (home['FTHG'].sum() + away['FTAG'].sum()) / max(total, 1)
        conceded = (home['FTAG'].sum() + away['FTHG'].sum()) / max(total, 1)
    return scored, conceded

def calc_shot_accuracy(df, team, n=5, loc='home'):
    """NEW: Shot accuracy = shots on target / total shots"""
    if loc == 'home':
        matches = df[df['HomeTeam'] == team].tail(n)
        if 'HST' in matches.columns and 'HS' in matches.columns:
            total_shots = matches['HS'].sum()
            on_target = matches['HST'].sum()
            return on_target / max(total_shots, 1)
    else:
        matches = df[df['AwayTeam'] == team].tail(n)
        if 'AST' in matches.columns and 'AS' in matches.columns:
            total_shots = matches['AS'].sum()
            on_target = matches['AST'].sum()
            return on_target / max(total_shots, 1)
    return 0.33  # Default if no data

def calc_corners_avg(df, team, n=5, loc='home'):
    """NEW: Average corners per game"""
    if loc == 'home':
        matches = df[df['HomeTeam'] == team].tail(n)
        if 'HC' in matches.columns:
            return matches['HC'].mean()
    else:
        matches = df[df['AwayTeam'] == team].tail(n)
        if 'AC' in matches.columns:
            return matches['AC'].mean()
    return 5.0  # Default

def calc_discipline(df, team, n=5, loc='home'):
    """NEW: Average yellow cards (discipline indicator)"""
    if loc == 'home':
        matches = df[df['HomeTeam'] == team].tail(n)
        if 'HY' in matches.columns:
            return matches['HY'].mean()
    else:
        matches = df[df['AwayTeam'] == team].tail(n)
        if 'AY' in matches.columns:
            return matches['AY'].mean()
    return 2.0  # Default

def engineer_features(df, window=5):
    """Enhanced feature engineering with stats"""
    features = []
    
    print(f"\nEngineering ENHANCED features...")
    print("Using: form, goals, shot accuracy, corners, discipline")
    
    for idx in range(window, len(df)):
        if idx % 200 == 0:
            print(f"  Processed {idx - window} matches...")
        
        match = df.iloc[idx]
        history = df.iloc[:idx]
        
        home = match['HomeTeam']
        away = match['AwayTeam']
        
        # Basic features
        h_form = calc_form(history, home, window)
        a_form = calc_form(history, away, window)
        h_gs, h_gc = calc_goals(history, home, window, 'home')
        a_gs, a_gc = calc_goals(history, away, window, 'away')
        
        # NEW: Advanced stats
        h_shot_acc = calc_shot_accuracy(history, home, window, 'home')
        a_shot_acc = calc_shot_accuracy(history, away, window, 'away')
        h_corners = calc_corners_avg(history, home, window, 'home')
        a_corners = calc_corners_avg(history, away, window, 'away')
        h_discipline = calc_discipline(history, home, window, 'home')
        a_discipline = calc_discipline(history, away, window, 'away')
        
        features.append({
            'HomeTeam': home,
            'AwayTeam': away,
            'home_form': h_form,
            'away_form': a_form,
            'home_goals_scored': h_gs,
            'away_goals_scored': a_gs,
            'home_goals_conceded': h_gc,
            'away_goals_conceded': a_gc,
            'home_shot_accuracy': h_shot_acc,
            'away_shot_accuracy': a_shot_acc,
            'home_corners_avg': h_corners,
            'away_corners_avg': a_corners,
            'home_discipline': h_discipline,
            'away_discipline': a_discipline,
            'form_diff': h_form - a_form,
            'FTR': match['FTR'],
            'Over2.5': match['Over2.5'],
            'BTTS': match['BTTS']
        })
    
    print(f"Created {len(features)} samples with {len(features[0])-3} features")
    return pd.DataFrame(features)

# ============================================================================
# MODEL TRAINING (Enhanced)
# ============================================================================

def train_models(features_df):
    print("\n" + "=" * 80)
    print("TRAINING MODELS (Enhanced Features)")
    print("=" * 80)
    
    feat_cols = [
        'home_form', 'away_form',
        'home_goals_scored', 'away_goals_scored',
        'home_goals_conceded', 'away_goals_conceded',
        'home_shot_accuracy', 'away_shot_accuracy',
        'home_corners_avg', 'away_corners_avg',
        'home_discipline', 'away_discipline',
        'form_diff'
    ]
    
    X = features_df[feat_cols]
    
    split_idx = int(len(X) * 0.8)
    X_train = X.iloc[:split_idx]
    X_test = X.iloc[split_idx:]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    models = {}
    
    # Match Result
    print("\n1. Match Result (1X2)")
    y_train = features_df.iloc[:split_idx]['FTR']
    y_test = features_df.iloc[split_idx:]['FTR']
    
    m = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
    m.fit(X_train_scaled, y_train)
    
    acc = accuracy_score(y_test, m.predict(X_test_scaled))
    print(f"   Accuracy: {acc:.2%}")
    
    # Feature importance
    importances = sorted(zip(feat_cols, m.feature_importances_), 
                        key=lambda x: x[1], reverse=True)[:5]
    print("   Top features:", ", ".join([f"{feat}" for feat, imp in importances]))
    
    models['result'] = m
    
    # Over/Under
    print("\n2. Over/Under 2.5")
    y_train = features_df.iloc[:split_idx]['Over2.5']
    y_test = features_df.iloc[split_idx:]['Over2.5']
    
    m = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42)
    m.fit(X_train_scaled, y_train)
    
    acc = accuracy_score(y_test, m.predict(X_test_scaled))
    print(f"   Accuracy: {acc:.2%}")
    models['over'] = m
    
    # BTTS
    print("\n3. BTTS")
    y_train = features_df.iloc[:split_idx]['BTTS']
    y_test = features_df.iloc[split_idx:]['BTTS']
    
    m = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42)
    m.fit(X_train_scaled, y_train)
    
    acc = accuracy_score(y_test, m.predict(X_test_scaled))
    print(f"   Accuracy: {acc:.2%}")
    models['btts'] = m
    
    print("\n" + "=" * 80)
    
    return models, scaler, feat_cols

# ============================================================================
# PREDICTION WITH MARKET COMPARISON
# ============================================================================

def predict_match(models, scaler, feat_cols, fixture, historical_df):
    """Enhanced prediction with market comparison"""
    
    home = fixture['HomeTeam']
    away = fixture['AwayTeam']
    
    # Calculate all features
    h_form = calc_form(historical_df, home, CONFIG['form_window'])
    a_form = calc_form(historical_df, away, CONFIG['form_window'])
    h_gs, h_gc = calc_goals(historical_df, home, CONFIG['form_window'], 'home')
    a_gs, a_gc = calc_goals(historical_df, away, CONFIG['form_window'], 'away')
    h_shot_acc = calc_shot_accuracy(historical_df, home, CONFIG['form_window'], 'home')
    a_shot_acc = calc_shot_accuracy(historical_df, away, CONFIG['form_window'], 'away')
    h_corners = calc_corners_avg(historical_df, home, CONFIG['form_window'], 'home')
    a_corners = calc_corners_avg(historical_df, away, CONFIG['form_window'], 'away')
    h_discipline = calc_discipline(historical_df, home, CONFIG['form_window'], 'home')
    a_discipline = calc_discipline(historical_df, away, CONFIG['form_window'], 'away')
    
    feats = {
        'home_form': h_form,
        'away_form': a_form,
        'home_goals_scored': h_gs,
        'away_goals_scored': a_gs,
        'home_goals_conceded': h_gc,
        'away_goals_conceded': a_gc,
        'home_shot_accuracy': h_shot_acc,
        'away_shot_accuracy': a_shot_acc,
        'home_corners_avg': h_corners,
        'away_corners_avg': a_corners,
        'home_discipline': h_discipline,
        'away_discipline': a_discipline,
        'form_diff': h_form - a_form
    }
    
    X = pd.DataFrame([feats])[feat_cols]
    X_scaled = scaler.transform(X)
    
    # Model predictions
    result_proba = models['result'].predict_proba(X_scaled)[0]
    classes = models['result'].classes_
    probs = dict(zip(classes, result_proba))
    
    h_prob = probs.get('H', 0.33)
    d_prob = probs.get('D', 0.33)
    a_prob = probs.get('A', 0.33)
    
    over_prob = models['over'].predict_proba(X_scaled)[0][1]
    btts_prob = models['btts'].predict_proba(X_scaled)[0][1]
    
    # Market comparison
    market_probs = None
    if fixture.get('odds'):
        market_probs = calc_implied_probs(fixture['odds'])
    
    return {
        'home': home,
        'away': away,
        'competition': fixture.get('Competition', 'Unknown'),
        'kickoff': fixture.get('Kickoff', ''),
        'probabilities': {
            'home': h_prob,
            'draw': d_prob,
            'away': a_prob,
            'over_2.5': over_prob,
            'btts_yes': btts_prob
        },
        'market_probs': market_probs,
        'stats': {
            'home_shot_acc': h_shot_acc,
            'away_shot_acc': a_shot_acc,
            'home_form': h_form,
            'away_form': a_form
        }
    }

# ============================================================================
# GUI (Enhanced)
# ============================================================================

class EducationalPredictorGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Educational Football Predictor v5.0 - Enhanced Edition")
        self.root.geometry("1500x900")
        
        self.logger = PredictionLogger()
        self.models = None
        self.predictions = []
        
        self.show_disclaimer()
        self.create_widgets()
    
    def show_disclaimer(self):
        popup = tk.Toplevel(self.root)
        popup.title("Educational Tool v5.0")
        popup.geometry("700x500")
        
        text = scrolledtext.ScrolledText(popup, wrap=tk.WORD, font=('Courier', 9))
        text.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        text.insert('1.0', DISCLAIMER)
        text.config(state=tk.DISABLED)
        
        ttk.Button(popup, text="I Understand", 
                  command=popup.destroy).pack(pady=10)
    
    def create_widgets(self):
        main = ttk.Frame(self.root, padding="10")
        main.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Title
        title_frame = ttk.Frame(main)
        title_frame.grid(row=0, column=0, columnspan=2, pady=10)
        
        ttk.Label(title_frame, text="Educational Football Predictor v5.0", 
                 font=('Arial', 18, 'bold')).pack()
        ttk.Label(title_frame, text="Enhanced with Market Comparison", 
                 font=('Arial', 10, 'italic')).pack()
        
        # Controls
        ctrl = ttk.LabelFrame(main, text="Controls", padding="10")
        ctrl.grid(row=1, column=0, columnspan=2, sticky=(tk.W, tk.E), pady=10)
        
        ttk.Label(ctrl, text="League:").grid(row=0, column=0, padx=5)
        self.league_var = tk.StringVar(value="Premier League")
        ttk.Combobox(ctrl, textvariable=self.league_var, 
                    values=["Premier League", "Top 5 European"], 
                    width=18).grid(row=0, column=1, padx=5)
        
        ttk.Button(ctrl, text="Update Data", 
                  command=self.update_data).grid(row=0, column=2, padx=5)
        ttk.Button(ctrl, text="Load & Train", 
                  command=self.train).grid(row=0, column=3, padx=5)
        ttk.Button(ctrl, text="Log Outcomes", 
                  command=self.log_outcomes).grid(row=0, column=4, padx=5)
        
        # Input panel
        input_frame = ttk.Frame(main)
        input_frame.grid(row=2, column=0, sticky=(tk.W, tk.E, tk.N, tk.S), padx=(0, 5))
        
        ttk.Label(input_frame, text="Paste Fixtures", 
                 font=('Arial', 10, 'bold')).pack(anchor=tk.W)
        
        self.fixture_text = scrolledtext.ScrolledText(input_frame, height=10, width=60, 
                                                     font=('Consolas', 9))
        self.fixture_text.pack(fill=tk.BOTH, expand=True, pady=5)
        self.fixture_text.insert('1.0', "Arsenal vs West Ham | Premier League | 17:00")
        
        ttk.Button(input_frame, text="Parse Fixtures", 
                  command=self.parse_fixtures).pack(fill=tk.X, pady=5)
        
        ttk.Label(input_frame, text="Paste Odds (Optional)", 
                 font=('Arial', 10, 'bold')).pack(anchor=tk.W, pady=(10, 0))
        
        self.odds_text = scrolledtext.ScrolledText(input_frame, height=8, width=60, 
                                                  font=('Consolas', 9))
        self.odds_text.pack(fill=tk.BOTH, expand=True, pady=5)
        self.odds_text.insert('1.0', "Arsenal vs West Ham: 1.45|4.50|6.00|1.80|1.70")
        
        ttk.Button(input_frame, text="Parse Odds", 
                  command=self.parse_odds).pack(fill=tk.X, pady=5)
        
        self.predict_btn = ttk.Button(input_frame, text="Generate Predictions", 
                                     command=self.predict, state=tk.DISABLED)
        self.predict_btn.pack(fill=tk.X, pady=10)
        
        # Main tabs
        notebook = ttk.Notebook(main)
        notebook.grid(row=2, column=1, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Predictions tab
        pred_frame = ttk.Frame(notebook)
        notebook.add(pred_frame, text="Predictions")
        
        self.tree = ttk.Treeview(pred_frame, 
                                columns=('Home', 'Away', 'H%', 'D%', 'A%', 'MktH%', 'Diff'),
                                show='headings', height=20)
        
        for col, width in [('Home', 130), ('Away', 130), ('H%', 60), ('D%', 60), 
                          ('A%', 60), ('MktH%', 70), ('Diff', 60)]:
            self.tree.heading(col, text=col)
            self.tree.column(col, width=width)
        
        scroll = ttk.Scrollbar(pred_frame, orient=tk.VERTICAL, command=self.tree.yview)
        self.tree.configure(yscroll=scroll.set)
        
        self.tree.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        scroll.grid(row=0, column=1, sticky=(tk.N, tk.S))
        
        # Dashboard tab
        dash_frame = ttk.Frame(notebook)
        notebook.add(dash_frame, text="Model vs Market")
        
        self.dash_text = scrolledtext.ScrolledText(dash_frame, height=25, width=90, 
                                                  font=('Consolas', 10))
        self.dash_text.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        self.update_dashboard()
        
        # History tab
        log_frame = ttk.Frame(notebook)
        notebook.add(log_frame, text="History")
        
        self.log_text = scrolledtext.ScrolledText(log_frame, height=25, width=90, 
                                                 font=('Consolas', 9))
        self.log_text.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        self.update_log_history()
        
        # Educational notes
        notes_frame = ttk.Frame(notebook)
        notebook.add(notes_frame, text="Learning Notes")
        
        notes_text = scrolledtext.ScrolledText(notes_frame, height=25, width=90, 
                                              font=('Consolas', 9), wrap=tk.WORD)
        notes_text.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        notes_text.insert('1.0', self.get_notes())
        notes_text.config(state=tk.DISABLED)
        
        # Status
        self.status = tk.StringVar(value="Ready. Load data to begin.")
        ttk.Label(main, textvariable=self.status, relief=tk.SUNKEN, 
                 anchor=tk.W).grid(row=3, column=0, columnspan=2, sticky=(tk.W, tk.E))
        
        # Configure weights
        self.root.columnconfigure(0, weight=1)
        self.root.rowconfigure(0, weight=1)
        main.columnconfigure(1, weight=2)
        main.columnconfigure(0, weight=1)
        main.rowconfigure(2, weight=1)
    
    def update_data(self):
        self.status.set("Updating...")
        self.root.update()
        
        try:
            leagues = ['E0'] if self.league_var.get() == "Premier League" else ['E0', 'SP1', 'D1', 'I1', 'F1']
            update_current_season(leagues)
            self.status.set("Updated")
            messagebox.showinfo("Success", "Data updated")
        except Exception as e:
            messagebox.showerror("Error", str(e))
    
    def train(self):
        self.status.set("Loading data...")
        self.root.update()
        
        try:
            leagues = ['E0'] if self.league_var.get() == "Premier League" else ['E0', 'SP1', 'D1', 'I1', 'F1']
            
            df = download_data(leagues, CONFIG['seasons'])
            df = preprocess_data(df)
            self.historical_df = df
            
            self.status.set("Training models...")
            self.root.update()
            
            features = engineer_features(df)
            self.models, self.scaler, self.feat_cols = train_models(features)
            
            self.status.set("Models trained!")
            self.predict_btn.config(state=tk.NORMAL)
            messagebox.showinfo("Success", "Models ready with enhanced features!")
        except Exception as e:
            self.status.set(f"Error: {e}")
            messagebox.showerror("Error", str(e))
    
    def parse_fixtures(self):
        text = self.fixture_text.get('1.0', tk.END)
        fixtures = parse_fixtures(text)
        
        if fixtures:
            self.fixtures = fixtures
            self.status.set(f"Parsed {len(fixtures)} fixtures")
        else:
            messagebox.showwarning("Warning", "No fixtures found")
    
    def parse_odds(self):
        if not hasattr(self, 'fixtures'):
            messagebox.showwarning("Warning", "Parse fixtures first")
            return
        
        text = self.odds_text.get('1.0', tk.END)
        self.fixtures = parse_odds(text, self.fixtures)
        
        with_odds = sum(1 for f in self.fixtures if f.get('odds'))
        self.status.set(f"Odds: {with_odds}/{len(self.fixtures)}")
    
    def predict(self):
        if not self.models:
            messagebox.showwarning("Warning", "Train models first")
            return
        
        if not hasattr(self, 'fixtures'):
            messagebox.showwarning("Warning", "Parse fixtures first")
            return
        
        stats = self.logger.get_stats()
        if stats['needs_more'] > 0:
            response = messagebox.askyesno(
                "Logging Required",
                f"Need {stats['needs_more']} more logs. Continue anyway?"
            )
            if not response:
                return
        
        self.status.set("Predicting...")
        self.root.update()
        
        try:
            for item in self.tree.get_children():
                self.tree.delete(item)
            
            self.predictions = []
            
            for fixture in self.fixtures:
                pred = predict_match(self.models, self.scaler, self.feat_cols, 
                                   fixture, self.historical_df)
                self.predictions.append(pred)
                
                p = pred['probabilities']
                mkt = pred['market_probs']
                
                if mkt:
                    mkt_h = f"{mkt['home_norm']:.1%}"
                    diff = f"{(p['home'] - mkt['home_norm']) * 100:+.1f}%"
                else:
                    mkt_h = "N/A"
                    diff = "N/A"
                
                self.tree.insert('', 'end', values=(
                    pred['home'], pred['away'],
                    f"{p['home']:.1%}", f"{p['draw']:.1%}", f"{p['away']:.1%}",
                    mkt_h, diff
                ))
            
            self.status.set(f"Predicted {len(self.predictions)} matches")
        except Exception as e:
            messagebox.showerror("Error", str(e))
    
    def log_outcomes(self):
        if not self.predictions:
            messagebox.showinfo("Info", "Generate predictions first")
            return
        
        dialog = tk.Toplevel(self.root)
        dialog.title("Log Outcomes")
        dialog.geometry("550x450")
        
        ttk.Label(dialog, text="Log Actual Results", 
                 font=('Arial', 12, 'bold')).pack(pady=10)
        
        for pred in self.predictions[:5]:
            frame = ttk.LabelFrame(dialog, text=f"{pred['home']} vs {pred['away']}", 
                                  padding="5")
            frame.pack(fill=tk.X, padx=10, pady=5)
            
            result_var = tk.StringVar(value="H")
            ttk.Radiobutton(frame, text="Home Win", variable=result_var, 
                          value="H").pack(side=tk.LEFT, padx=5)
            ttk.Radiobutton(frame, text="Draw", variable=result_var, 
                          value="D").pack(side=tk.LEFT, padx=5)
            ttk.Radiobutton(frame, text="Away Win", variable=result_var, 
                          value="A").pack(side=tk.LEFT, padx=5)
            
            ttk.Button(frame, text="Log Result", 
                      command=lambda p=pred, r=result_var: self.save_log(p, r.get(), dialog)
                      ).pack(side=tk.RIGHT, padx=5)
        
        ttk.Button(dialog, text="Close", command=dialog.destroy).pack(pady=10)
    
    def save_log(self, pred, result, dialog):
        self.logger.add_log(pred, {'result': result})
        self.update_dashboard()
        self.update_log_history()
        self.status.set(f"Logged. Total: {len(self.logger.logs)}")
        messagebox.showinfo("Logged", "Outcome recorded for analysis")
        dialog.destroy()
    
    def update_dashboard(self):
        self.dash_text.delete('1.0', tk.END)
        
        stats = self.logger.get_stats()
        
        self.dash_text.insert(tk.END, "=" * 85 + "\n")
        self.dash_text.insert(tk.END, "MODEL vs MARKET COMPARISON DASHBOARD\n")
        self.dash_text.insert(tk.END, "=" * 85 + "\n\n")
        
        if stats['total'] == 0:
            self.dash_text.insert(tk.END, "No logs yet. Generate predictions and log outcomes.\n\n")
            self.dash_text.insert(tk.END, "This dashboard will show:\n")
            self.dash_text.insert(tk.END, "• Your model accuracy vs bookmaker accuracy\n")
            self.dash_text.insert(tk.END, "• Why markets are hard to beat\n")
            self.dash_text.insert(tk.END, "• Bookmaker margin analysis\n")
        else:
            self.dash_text.insert(tk.END, f"TOTAL LOGGED PREDICTIONS: {stats['total']}\n\n")
            
            self.dash_text.insert(tk.END, "MODEL PERFORMANCE:\n")
            self.dash_text.insert(tk.END, f"  Correct: {stats['model_correct']}/{stats['total']}\n")
            self.dash_text.insert(tk.END, f"  Accuracy: {stats['model_accuracy']:.2%}\n\n")
            
            if stats['market_available'] > 0:
                self.dash_text.insert(tk.END, "MARKET PERFORMANCE:\n")
                self.dash_text.insert(tk.END, f"  Correct: {stats['market_correct']}/{stats['market_available']}\n")
                self.dash_text.insert(tk.END, f"  Accuracy: {stats['market_accuracy']:.2%}\n\n")
                
                diff = stats['model_accuracy'] - stats['market_accuracy']
                self.dash_text.insert(tk.END, "COMPARISON:\n")
                if abs(diff) < 0.02:
                    self.dash_text.insert(tk.END, f"  Similar performance ({diff:+.1%} difference)\n")
                elif diff > 0:
                    self.dash_text.insert(tk.END, f"  Model beating market by {diff:.1%}!\n")
                    self.dash_text.insert(tk.END, f"  (Need 100+ logs to confirm)\n")
                else:
                    self.dash_text.insert(tk.END, f"  Market beating model by {-diff:.1%}\n")
                    self.dash_text.insert(tk.END, f"  (This is typical - markets are efficient)\n")
            
            self.dash_text.insert(tk.END, "\n" + "-" * 85 + "\n")
            self.dash_text.insert(tk.END, "STATISTICAL SIGNIFICANCE:\n\n")
            
            if stats['total'] < 30:
                self.dash_text.insert(tk.END, f"⚠ Sample too small ({stats['total']} < 30)\n")
                self.dash_text.insert(tk.END, f"   Need {30 - stats['total']} more logs\n")
            elif stats['total'] < 100:
                self.dash_text.insert(tk.END, f"📊 Decent sample ({stats['total']})\n")
                self.dash_text.insert(tk.END, f"   Target: 100+ for confidence\n")
            else:
                self.dash_text.insert(tk.END, f"✓ Good sample size ({stats['total']})\n")
                self.dash_text.insert(tk.END, f"   Results meaningful\n")
            
            self.dash_text.insert(tk.END, "\n" + "-" * 85 + "\n")
            self.dash_text.insert(tk.END, "KEY LEARNING:\n")
            self.dash_text.insert(tk.END, "• Bookmaker margins (5-7%) make beating odds hard\n")
            self.dash_text.insert(tk.END, "• Markets incorporate info faster than models\n")
            self.dash_text.insert(tk.END, "• Even 55% accuracy doesn't guarantee profit\n")
    
    def update_log_history(self):
        self.log_text.delete('1.0', tk.END)
        
        if not self.logger.logs:
            self.log_text.insert(tk.END, "No logs yet\n")
            return
        
        self.log_text.insert(tk.END, "=" * 85 + "\n")
        self.log_text.insert(tk.END, "PREDICTION LOG HISTORY\n")
        self.log_text.insert(tk.END, "=" * 85 + "\n\n")
        
        for i, log in enumerate(reversed(self.logger.logs[-20:]), 1):
            date = datetime.fromisoformat(log['timestamp']).strftime('%Y-%m-%d')
            model_ok = "✓" if log.get('model_correct') else "✗"
            market_ok = "✓" if log.get('market_correct') == True else ("✗" if log.get('market_correct') == False else "?")
            
            self.log_text.insert(tk.END, f"{i}. {date} - {log['match']}\n")
            self.log_text.insert(tk.END, f"   Model: {log['model_predicted']} {model_ok} | ")
            self.log_text.insert(tk.END, f"Market: {market_ok} | ")
            self.log_text.insert(tk.END, f"Actual: {log['actual']}\n\n")
    
    def get_notes(self):
        return """
═══════════════════════════════════════════════════════════════════════
                    EDUCATIONAL NOTES - v5.0 ENHANCED
═══════════════════════════════════════════════════════════════════════

NEW IN v5.0: ADVANCED FEATURES & MARKET COMPARISON
───────────────────────────────────────────────────────────────────────

1. ENHANCED FEATURES EXPLAINED
   
   Shot Accuracy = Shots on Target / Total Shots
   → Teams that hit the target more often score more goals
   
   Corners Average = Average corners per game
   → More corners = more attacking pressure = more goals
   
   Discipline = Average yellow cards
   → More cards = aggressive play style (can affect results)
   
   These features help the model understand team style, not just results!

2. MARKET COMPARISON - THE KEY LEARNING
   
   Your Model: Trained on historical stats
   Bookmakers: Use same data PLUS insider info, injuries, motivation
   
   Typical Results After 100 Logs:
   • Model accuracy: 52-55%
   • Market accuracy: 56-59%
   
   Why markets win:
   → Aggregate wisdom of thousands of bettors
   → Professional traders with better data
   → Real-time updates (injuries, team news)
   → Your model uses only past stats

3. BOOKMAKER MARGIN (OVERROUND)
   
   Fair 50/50 match: 2.00 / 2.00 odds (100% total probability)
   Actual odds: 1.91 / 1.91 (105% total = 5% margin)
   
   This 5-7% margin means:
   → Even if your model is accurate, you're fighting uphill
   → Need 53%+ accuracy just to break even
   → Long-term profit requires 58%+ (very rare)

4. FEATURE IMPORTANCE
   
   After training, check which features matter most:
   • Form: Usually #1 (recent results predict future)
   • Goals scored/conceded: #2-3
   • Shot accuracy: #4-5
   • Corners, cards: Minor impact
   
   This teaches: Simple features often beat complex ones!

5. CALIBRATION - CRITICAL CONCEPT
   
   Good Model: When it says 60% → wins 60% of time
   Bad Model: When it says 60% → wins 45% of time
   
   Markets are well-calibrated (60% predictions ≈ 60% wins)
   Your model likely needs 100+ logs to calibrate well

6. SAMPLE SIZE REALITY
   
   • 10 logs: Pure luck dominates
   • 30 logs: Start seeing patterns
   • 100 logs: Confidence in accuracy
   • 500 logs: Statistical significance
   • 1000+ logs: True model assessment
   
   Even great models have 20+ match losing streaks!

7. WHAT TO TRACK
   
   □ Model accuracy vs baseline (33% for 1X2)
   □ Model vs market accuracy (head-to-head)
   □ Calibration (do 70% predictions win 70%?)
   □ Feature importance (what predicts best?)
   □ Variance (losing streaks show randomness)

8. KELLY CRITERION (ADVANCED MATH)
   
   If you had genuine edge:
   Bet Size = (Edge / Odds) × Bankroll
   
   Example: 60% win prob at 2.00 odds
   Edge = 0.60 - 0.50 = 0.10 (10%)
   Kelly = 0.10 / 1.00 = 10% of bankroll
   
   But overestimate edge by 5% → bankruptcy!
   That's why markets are so dangerous.

9. WHY THIS TOOL EXISTS
   
   NOT to make money gambling
   
   BUT to learn:
   ✓ Machine learning model evaluation
   ✓ Feature engineering importance
   ✓ Why efficient markets are hard to beat
   ✓ Statistical significance and variance
   ✓ Real-world data science challenges

10. RESPONSIBLE APPROACH
    
    If you gamble with real money:
    • Max 1% bankroll per bet (never more)
    • Track EVERY outcome honestly
    • Accept markets usually win
    • Stop if stressful/problematic
    • Help: 1-800-522-4700

═══════════════════════════════════════════════════════════════════════
Remember: The goal is learning data science, not beating bookmakers.
Even professional betting syndicates struggle with 2-3% edges.
═══════════════════════════════════════════════════════════════════════
"""

def main():
    ensure_folders()
    
    if GUI_AVAILABLE:
        root = tk.Tk()
        app = EducationalPredictorGUI(root)
        root.mainloop()
    else:
        print("GUI not available")
        print(DISCLAIMER)

if __name__ == "__main__":
    main()


Updating current season...
  Premier League: 380 matches
  La Liga: 380 matches
  Bundesliga: 306 matches
  Serie A: 380 matches
  Ligue 1: 306 matches



Select option (1-3, default 1):  1



📝 MANUAL FIXTURE ENTRY
Enter matches in format: HomeTeam vs AwayTeam
Examples:
  - Arsenal vs Man City
  - Real Madrid vs Barcelona
  - Bayern Munich vs Dortmund

Type 'done' when finished, or press Enter to skip



Match 1:  Atalanta vs. Club Brugge


  ⚠️  Invalid format. Use: HomeTeam vs AwayTeam


Match 1:  Atalanta vs Club Brugge
Match 2:  Kairat Almaty vs Real Madrid
Match 3:  Atlético Madrid vs Eintracht Frankfurt
Match 4:  Chelsea vs Benfica
Match 5:  Galatasaray vs Liverpool
Match 6:  Inter Milan vs Slavia Prague
Match 7:  Marseille vs Ajax
Match 8:  Pafos vs Bayern Munich
Match 9:  Bodø/Glimt vs Tottenham
Match 10:  done



✓ Added 9 matches

STEP 5: PREDICTING TODAY'S MATCHES
Date: Tuesday, September 30, 2025

🏟️  Atalanta vs Club Brugge
    Competition: Manual Entry
--------------------------------------------------------------------------------
⚠️  WARNING: Limited historical data for these teams
   Predictions may be less reliable
📊 Match Result:
   Atalanta Win: 47.0%
   Draw:           8.0%
   Club Brugge Win: 45.0%

⚽ Goals:
   Over 2.5:  76.2%
   Under 2.5: 23.8%

🎯 BTTS:
   Yes: 64.1%
   No:  35.9%

🏟️  Kairat Almaty vs Real Madrid
    Competition: Manual Entry
--------------------------------------------------------------------------------
⚠️  WARNING: Limited historical data for these teams
   Predictions may be less reliable
📊 Match Result:
   Kairat Almaty Win: 47.0%
   Draw:           8.0%
   Real Madrid Win: 45.0%

⚽ Goals:
   Over 2.5:  76.2%
   Under 2.5: 23.8%

🎯 BTTS:
   Yes: 64.1%
   No:  35.9%

🏟️  Atlético Madrid vs Eintracht Frankfurt
    Competition: Manual Entry
-----------------